# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

FlyRank's clients have thousands of published pages each, and no editor has time to manually
re-check all of them every week. The useful output isn't a report about the whole inventory —
it's a short, ordered list of the pages worth an editor's next hour: which ones to refresh,
expand, protect, prune, or leave alone. That's a ranking/scoring problem sitting directly on top
of data FlyRank already has (impressions, clicks, position, freshness, search demand), which is
why I'm picking this lane over the others: the decision it improves is concrete, the actor is a
real person with a real backlog, and the starter dataset already carries the signals a score
would need. I'll confirm or swap this by end of Week 4 once I've audited the signals properly.

## 2. The question: decision, action, cost of a wrong call

**Question:** Which content items should be reviewed first for refresh, expansion, protection,
pruning, or monitoring?

**Decision it improves:** which pages an editor spends limited weekly review time on, out of the
full content inventory — not "will this page decline," but "where should attention go first."

**Who acts, and what they do:** a content/SEO editor works a ranked queue top-down. For each item
near the top they take one specific action (refresh the content, expand it, protect it as-is,
prune it, or just keep monitoring) rather than reading a general report.

**Cost of a wrong call:** the errors are not symmetric.
- A false positive (flagging a stable, healthy page as urgent) wastes editor hours that should
  have gone to a page that is actually losing traffic.
- A false negative (missing a page that is genuinely declining while demand still exists) means
  real clicks and revenue keep leaking for weeks before anyone notices by hand.
Because missing a real decliner is more expensive over time than reviewing an extra healthy page,
a useful score should lean toward not missing true decliners, even at some cost in wasted reviews.

**Why data/ML helps instead of a plain if-statement:** see the numbers below — a single-column
rule ("flag anything trending down") flags more than half the entire inventory, which gives an
editor no real priority order. Combining a few signals narrows that to something an editor could
plausibly work through in a week, and no single column does that narrowing on its own.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
n = len(df)
print(f"Rows: {n}, columns: {df.shape[1]}")

# Gotcha check first (from flyrank-data skill): avg_position == 0 means "no data", not rank zero
no_position_data = (df['avg_position'] == 0).sum()
print(f"\nRows with avg_position == 0 (no rank data, not rank #0): {no_position_data} "
      f"({no_position_data/n*100:.1f}%)")

# Number 1: a single-column rule is too broad to be a queue
down = (df['trend_direction'] == 'down').sum()
print(f"\nPages simply trending down: {down}/{n} = {down/n*100:.1f}%")
print("-> more than half the inventory. Not a usable weekly queue on its own.")

# Number 2: combining signals gives a plausible, actionable queue size
worth_reviewing = (
    (df['trend_direction'] == 'down')
    & (df['search_volume'] > 0)
    & (df['impressions_last_30d'] > 10)
).sum()
print(f"\nDown + real search demand + still getting impressions: "
      f"{worth_reviewing}/{n} = {worth_reviewing/n*100:.1f}%")
print("-> a narrower, more plausible weekly review set than the raw trend flag.")

# Number 3: a second angle - stale content that still has an audience (refresh candidates)
stale_but_visible = (
    df['freshness_tier'].isin(['91-180', '181+'])
    & (df['impressions_last_30d'] > 50)
).sum()
print(f"\nUntouched 90+ days but still pulling real traffic (>50 impressions/30d): "
      f"{stale_but_visible}/{n} = {stale_but_visible/n*100:.1f}%")
print("-> a concrete refresh candidate pool, distinct from the decline pool above.")

Rows: 30000, columns: 44

Rows with avg_position == 0 (no rank data, not rank #0): 1205 (4.0%)

Pages simply trending down: 16262/30000 = 54.2%
-> more than half the inventory. Not a usable weekly queue on its own.

Down + real search demand + still getting impressions: 6697/30000 = 22.3%
-> a narrower, more plausible weekly review set than the raw trend flag.

Untouched 90+ days but still pulling real traffic (>50 impressions/30d): 7056/30000 = 23.5%
-> a concrete refresh candidate pool, distinct from the decline pool above.


## 4. Careful words: what I can and can't claim

**What this work can claim:**
- **Observed** patterns within this 90-day trailing window, across the 32 clients in the starter
  data (or however many are in the warehouse release once I move to it).
- A **ranking/prioritization tool** — a way to sort limited editor attention, not a certainty
  machine.
- **Decision-support**: "review this one first" backed by reason codes, not "this page will
  decline."
- **Directional** relationships (e.g. stale + demand tends to co-occur with decline) — association,
  not mechanism.

**What this work will never claim:**
- That refreshing a flagged page **causes** it to recover. I only have observational data, no
  before/after experiment on an actual refresh.
- Anything about **why Google ranks pages the way it does**. `avg_position` and `trend_pct` are
  outcomes I observe, not evidence about a private ranking algorithm.
- That patterns found in this dataset **generalize** beyond these clients, this time window, and
  this industry mix.
- That a high score **guarantees** a wrong call was avoided — it lowers the odds, it doesn't
  remove them.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.